# Lecture 7 - Backdoor paths

McElreath's lectures for the whole book are available here: https://github.com/rmcelreath/stat_rethinking_2022

An R/Stan repo of code is available here: https://vincentarelbundock.github.io/rethinking2/

An excellent port to Python/PyMC Code is available here: https://github.com/dustinstansbury/statistical-rethinking-2023

You are encouraged to work through both of these versions to re-enforce what we're doing in class.

In [ ]:
# Load R packages
library(cmdstanr)    # R interface to Stan
library(posterior)   # Working with posterior draws
library(bayesplot)   # Plotting posterior draws
library(ggplot2)     # bayesplot figures are ggplots, so we can add to them

# Helper functions
stdize <- function(x) (x - mean(x))/sd(x)

indexall <- function(L){
    poo <- unique(L)
    Ix <- as.integer(factor(L, levels = poo))
    list(poo, Ix)
}

# Stan's sampler with the same defaults as pm.sample(1000): 4 chains, 1000 warmup, 1000 draws
sample_stan <- function(model, data){
    model$sample(data = data, chains = 4, parallel_chains = 4, iter_warmup = 1000,
                 iter_sampling = 1000, refresh = 0, show_exceptions = FALSE)
}

# Save the current figure to file (uncomment the savefig() calls below to use)
savefig <- function(file, width = 7, height = 5){
    dev.copy(jpeg, file, width = width, height = height, units = "in", res = 300)
    invisible(dev.off())
}

## Return to Waffle Houses

From the waffle house data we can assert a full causal graph:

![](wafflez.jpg)

And by importing the data, we can check the implications of our assumptions, and how playing with the elemental confounds changes the effect sizes we see in our statistical model.

In [ ]:
# Import data
ddata <- read.csv('WaffleDivorce.csv', sep = ";")
# Display top 5 rows
head(ddata, 5)

In [ ]:
# Standardize variables
AgeMarriage <- stdize(ddata$MedianAgeMarriage)
MarriageRate <- stdize(ddata$Marriage)
PropSlaves1860 <- stdize(ddata$PropSlaves1860)
Divorce <- stdize(ddata$Divorce)
WaffleHouses <- stdize(ddata$WaffleHouses)
South <- stdize(ddata$South)

With this diagram and data, let's see how blocking the backdor through conditioning on S versus A and M changes the effect of W on D.

In [ ]:
Smod_code <- "
data {
  int<lower=0> N;
  vector[N] WaffleHouses;
  vector[N] South;
  vector[N] Divorce;
}
parameters {
  real Intercept;                    // Baseline intercept
  real WaffleHouses_eff;             // Waffle House effect
  real South_eff;                    // Southern
  real<lower=0, upper=10> SD_obs;    // Error
}
model {
  // Priors
  Intercept ~ normal(0, 0.2);
  WaffleHouses_eff ~ normal(0, 0.5);
  South_eff ~ normal(0, 0.5);
  SD_obs ~ uniform(0, 10);

  // Linear model
  vector[N] mu = Intercept + WaffleHouses_eff*WaffleHouses + South_eff*South;

  // Likelihood
  Divorce ~ normal(mu, SD_obs);
}
"
Smod <- cmdstan_model(write_stan_file(Smod_code))

In [ ]:
AMmod_code <- "
data {
  int<lower=0> N;
  vector[N] WaffleHouses;
  vector[N] AgeMarriage;
  vector[N] MarriageRate;
  vector[N] Divorce;
}
parameters {
  real Intercept;                    // Baseline intercept
  real WaffleHouses_eff;             // Waffle House effect
  real AgeMarriage_eff;              // Marriage age
  real MarriageRate_eff;             // Marriage rate
  real<lower=0, upper=10> SD_obs;    // Error
}
model {
  // Priors
  Intercept ~ normal(0, 0.2);
  WaffleHouses_eff ~ normal(0, 0.5);
  AgeMarriage_eff ~ normal(0, 0.5);
  MarriageRate_eff ~ normal(0, 0.5);
  SD_obs ~ uniform(0, 10);

  // Linear model
  vector[N] mu = Intercept + WaffleHouses_eff*WaffleHouses + AgeMarriage_eff*AgeMarriage + MarriageRate_eff*MarriageRate;

  // Likelihood
  Divorce ~ normal(mu, SD_obs);
}
"
AMmod <- cmdstan_model(write_stan_file(AMmod_code))

The last two models each drop one of the two covariates. Rather than write out two more Stan programs, we can write **one** model with a single covariate alongside Waffle Houses, then feed it either marriage age or marriage rate:

In [ ]:
Xmod_code <- "
data {
  int<lower=0> N;
  vector[N] WaffleHouses;
  vector[N] X;                       // either marriage age or marriage rate
  vector[N] Divorce;
}
parameters {
  real Intercept;                    // Baseline intercept
  real WaffleHouses_eff;             // Waffle House effect
  real X_eff;                        // Effect of the covariate
  real<lower=0, upper=10> SD_obs;    // Error
}
model {
  // Priors
  Intercept ~ normal(0, 0.2);
  WaffleHouses_eff ~ normal(0, 0.5);
  X_eff ~ normal(0, 0.5);
  SD_obs ~ uniform(0, 10);

  // Linear model
  vector[N] mu = Intercept + WaffleHouses_eff*WaffleHouses + X_eff*X;

  // Likelihood
  Divorce ~ normal(mu, SD_obs);
}
"
Xmod <- cmdstan_model(write_stan_file(Xmod_code))

# Marriage age model (Amod) and marriage rate model (Mmod) are the same model, different data
Amod_data <- list(N = length(Divorce), WaffleHouses = WaffleHouses, X = AgeMarriage, Divorce = Divorce)
Mmod_data <- list(N = length(Divorce), WaffleHouses = WaffleHouses, X = MarriageRate, Divorce = Divorce)

In [ ]:
trace_s <- sample_stan(Smod, list(N = length(Divorce), WaffleHouses = WaffleHouses, South = South, Divorce = Divorce))
trace_am <- sample_stan(AMmod, list(N = length(Divorce), WaffleHouses = WaffleHouses, AgeMarriage = AgeMarriage,
                                    MarriageRate = MarriageRate, Divorce = Divorce))
trace_a <- sample_stan(Xmod, Amod_data)
trace_m <- sample_stan(Xmod, Mmod_data)

In [ ]:
# Look at southern model results
trace_s$summary()

In [ ]:
mcmc_intervals(trace_s$draws(c("Intercept", "WaffleHouses_eff", "South_eff", "SD_obs"))) + vline_0()
# ggsave('waffleS.jpg', dpi = 300)

In [ ]:
trace_am$summary()

In [ ]:
# Marriage age model (X_eff is the marriage age effect)
trace_a$summary()

In [ ]:
# Marriage rate model (X_eff is the marriage rate effect)
trace_m$summary()

In [ ]:
color_scheme_set("blue")

# Gather the Waffle House effect (and each model's other covariates) into one set of labelled draws
combo <- cbind(
    `Southern: WaffleHouses`     = as.vector(trace_s$draws("WaffleHouses_eff")),
    `Southern: South`            = as.vector(trace_s$draws("South_eff")),
    `Marriage A&M: WaffleHouses` = as.vector(trace_am$draws("WaffleHouses_eff")),
    `Marriage A&M: AgeMarriage`  = as.vector(trace_am$draws("AgeMarriage_eff")),
    `Marriage A&M: MarriageRate` = as.vector(trace_am$draws("MarriageRate_eff")),
    `Marriage M: WaffleHouses`   = as.vector(trace_m$draws("WaffleHouses_eff")),
    `Marriage M: MarriageRate`   = as.vector(trace_m$draws("X_eff"))
)

mcmc_intervals(combo) + vline_0() + ggtitle("Waffle House effects model")

Humm, we can see that South negates the Waffle House effect as expected, however conditioning on M and A doesn't do the same thing...there must be other, unmeasured backdoor paths through W to have an effect. 